# Course 5: Winning Online AI Competitions
## Session 1: How competitions work, and a first submission

**Goal:** go from the data files to a valid submission twice: a constant baseline, then a simple model.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on synthetic or built-in data. Cells labelled NOT RUN need a Kaggle account, a competition's data, a GPU or an extra library.

### 1. The three files every tabular competition gives you
`train.csv` has features and the target. `test.csv` has features only. `sample_submission.csv` shows the exact format expected. This cell writes small synthetic versions so the rest of the notebook runs anywhere.

In [ ]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
def make_tabular(n=6000, seed=0):
    """A synthetic binary task with numeric columns, one high-cardinality category, groups and a date."""
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({"id": np.arange(n)})
    for j in range(6): df[f"num{j}"] = rng.normal(size=n)
    n_shops = 300
    shop_effect = rng.normal(scale=1.0, size=n_shops)
    df["shop"] = rng.zipf(1.5, n).clip(max=n_shops) - 1                         # a few big shops, many small ones
    df["customer"] = rng.integers(0, n // 8, n)                                 # about 8 rows per customer
    cust_effect = rng.normal(scale=1.2, size=n // 8)
    df["day"] = np.sort(rng.integers(0, 365, n))
    logit = 0.8 * df.num0 - 0.6 * df.num1 + 0.5 * df.num2 * df.num3 + shop_effect[df["shop"]] + cust_effect[df.customer] + 0.004 * df.day - 1.0
    df["target"] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    return df
NUM = [f"num{j}" for j in range(6)]

In [ ]:
import os
df = make_tabular(3000)
os.makedirs("data", exist_ok=True)
train, test = df.iloc[:2000].copy(), df.iloc[2000:].copy()
hidden = test[["id", "target"]].copy()                           # on Kaggle you never see this
train.to_csv("data/train.csv", index=False); test.drop(columns="target").to_csv("data/test.csv", index=False)
pd.DataFrame({"id": test.id, "target": 0.5}).to_csv("data/sample_submission.csv", index=False)
train, test, sample = pd.read_csv("data/train.csv"), pd.read_csv("data/test.csv"), pd.read_csv("data/sample_submission.csv")
print(train.shape, test.shape, sample.shape); print(sample.head(3))

### 2. Look before you model
Five checks that take one minute and prevent most first-day mistakes.

In [ ]:
print("1. target mean:", round(train.target.mean(), 3))
print("2. missing values:", int(train.isna().sum().sum()))
print("3. columns in train but not in test:", sorted(set(train.columns) - set(test.columns)))
print("4. ids overlap between train and test:", len(set(train.id) & set(test.id)))
print("5. test ids match the sample submission, in order:", bool((test.id.values == sample.id.values).all()))

### 3. Submission one: a constant
Predict the training mean for every row. It sets the floor: any model must beat it, and it proves your file format is right before you spend time on features.

In [ ]:
from sklearn.metrics import roc_auc_score, log_loss
def validate_submission(sub, sample):
    assert list(sub.columns) == list(sample.columns), "wrong columns"
    assert len(sub) == len(sample) and (sub.id.values == sample.id.values).all(), "ids differ from the sample submission"
    assert sub.target.notna().all() and sub.target.between(0, 1).all(), "predictions must be probabilities with no missing values"
    return True
def score(sub): return log_loss(hidden.target, sub.target), roc_auc_score(hidden.target, sub.target)     # the organisers do this, not you

sub0 = sample.copy(); sub0["target"] = train.target.mean()
validate_submission(sub0, sample); sub0.to_csv("submission_constant.csv", index=False)
print("constant baseline: log loss %.4f  AUC %.3f" % score(sub0))

### 4. Submission two: a simple model with a local validation score
Hold out part of the training data, check the score there, and only then predict the test set. Write down both numbers: the gap between your local score and the leaderboard is the most useful number in a competition.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
tr, va = train_test_split(train, test_size=0.25, random_state=0, stratify=train.target)
model = LogisticRegression(max_iter=1000).fit(tr[NUM], tr.target)
local = log_loss(va.target, model.predict_proba(va[NUM])[:, 1])
sub1 = sample.copy(); sub1["target"] = model.predict_proba(test[NUM])[:, 1]
validate_submission(sub1, sample); sub1.to_csv("submission_logreg.csv", index=False)
lb = score(sub1)
print(f"logistic regression: local log loss {local:.4f} | 'leaderboard' log loss {lb[0]:.4f} | AUC {lb[1]:.3f}")
assert lb[0] < score(sub0)[0]

### 5. An experiment log from day one

In [ ]:
log = pd.DataFrame([
    {"n": 1, "what": "constant = train mean", "local": None, "leaderboard": round(score(sub0)[0], 4), "file": "submission_constant.csv"},
    {"n": 2, "what": "logistic regression on 6 numeric columns", "local": round(local, 4), "leaderboard": round(lb[0], 4), "file": "submission_logreg.csv"}])
log.to_csv("experiments.csv", index=False); print(log.to_string(index=False))

### 6. The Kaggle API (NOT RUN here)

In [ ]:
# NOT RUN: needs a Kaggle account, an API token, and the competition rules accepted on the website.
# Setup once: create a token on your Kaggle account page and follow the kaggle package's README for where to put it.
#   pip install kaggle
#   kaggle competitions list
#   kaggle competitions download -c playground-series-sXeY -p data         # pick a current Playground competition
#   unzip -o data/*.zip -d data
#   kaggle competitions submit -c playground-series-sXeY -f submission_constant.csv -m "constant baseline"
#   kaggle competitions submissions -c playground-series-sXeY
# Rules to read before anything else: the daily submission limit, team size, external data, and whether it is a code competition.

### TODO
1. Join a Getting Started or Playground competition and submit a constant, then a simple model. Record local and leaderboard scores.
2. Break your submission on purpose in three ways (shuffled ids, a missing row, a value of 1.2) and confirm `validate_submission` catches each one.
3. Read the competition's evaluation page and write the metric as a formula.